# 🧠 DietBot: PDF-Grounded Question Generation and Evaluation

This notebook generates realistic, emotionally authentic questions grounded in authoritative PDF content (e.g., ADA guidelines), and verifies whether "answerable" questions are actually supported by the reference documents using a retrieval-augmented LLM.

---

## 📌 What It Does

1. **Ingests** all PDF files from `data/input_pdfs/`
2. **Chunks & Embeds** documents using LangChain + FAISS
3. **Generates** 100 natural-language questions (50 answerable, 50 unanswerable) using GPT-4o
4. **Verifies** answerable questions using GPT-4o against the original documents
5. **Saves** results to `data/csv_outputs/`

## 🚀 How to Use

1. **Set your OpenAI API key**

Create a `.env` file in the `notebooks/` directory with:

```env
OPENAI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
```

Run cells in order

In [ ]:
!pip install pandas
!pip install --upgrade langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.68
    Uninstalling langchain-core-0.3.68:
      Successfully uninstalled langchain-core-0.3.68
  Attempting uninstall: langchain-text-splitters 0/3 [langchain-core]
    Found existing installation: langchain-text-splitters 0.3.8langchain-core]
    Uninstalling langchain-text-splitters-0.3.8: 0/3 [langchain-core]
      Successfully uninstalled langchain-text-splitters-0.3.8 [langchain-core]
  Attempting uninstall: langchain━━━━━━━━━━━ 0/3 [langchain-core]
    Found existing installation: langchain 0.3.260/3 [langchain-core]
    Uninstalling langchain-0.3.26:╸━━━━━━━━━━━━━ 2/3 [langchain]
      Successfully uninstalled langchain-0.3.260m━━━━━━━━━━━━━ 2/3 [langchain]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain]/3 [langchain]


### 🧠 Run a Local LLM with Ollama (Gemma 3)

This cell sends a prompt to a **locally running Ollama instance**, using the `requests` library to call the `/api/generate` endpoint.

**What it does:**
- Connects to `http://localhost:11434`
- Sends a sample health-related prompt to the `gemma3:4b-it-qat` model:
  > _"Can you manage diabetes just with food and no meds?"_
- If the call succeeds, you'll see the model's response printed below.

#### ⚙️ Before you run this:
- Make sure you have [Ollama](https://ollama.com) installed and running locally
- You should have the model pulled (e.g., `gemma3:4b-it-qat`)  
  Pull it by running:
  ```bash
  ollama pull gemma3:4b-it-qat
  ```
- If you're using a remote or hosted Ollama server, **update the `OLLAMA_API_URL`** to match your endpoint

✅ If everything works, you’ll see a generated answer to the question printed in the cell output.


In [8]:
import requests

OLLAMA_API_URL = "http://localhost:11434/api/generate"

payload = {
    "model": "gemma3:4b-it-qat",  # or whatever model you pulled
    "prompt": "Can you manage diabetes just with food and no meds?",
    "stream": False
}

response = requests.post(OLLAMA_API_URL, json=payload)

if response.ok:
    result = response.json()
    print(result["response"])
else:
    print(f"Error: {response.status_code} - {response.text}")

Okay, let's tackle this important and complex question. The short answer is: **it *can* be managed, but it's incredibly challenging and not suitable for everyone.** It's definitely possible to manage type 2 diabetes with diet and lifestyle changes alone, but it requires a huge commitment, careful planning, and ongoing monitoring. Here's a breakdown of what’s involved and why it's not a guaranteed solution:

**1. Type 2 Diabetes and the Role of Food & Lifestyle:**

* **Insulin Resistance:** Type 2 diabetes is largely driven by insulin resistance – where your body's cells don't respond effectively to insulin. Insulin is the hormone that helps glucose (sugar) from your blood enter your cells to be used for energy.
* **Diet's Impact:** Diet plays a *massive* role in controlling insulin resistance.  
    * **Carbohydrates:** Managing carb intake is crucial. Focusing on complex carbs (whole grains, legumes, vegetables) over simple carbs (sugary drinks, white bread) helps stabilize blood suga

### ✅ Manually Curated 100 Diabetes Questions

This cell creates a clean and balanced dataset of 100 user-style questions about diabetes, split evenly into:

- **50 Answerable Questions**:  
  These are natural, emotionally honest questions based on common concerns about food, lifestyle, and blood sugar.  
  They reflect what real people might ask a chatbot when trying to manage their diabetes or prediabetes.

- **50 Unanswerable Questions**:  
  These are speculative, misinformed, or pseudoscientific — things people might wonder or read online, but that **aren’t supported** by clinical sources like the ADA guidelines.

#### 🛠️ What the Code Does:
- Assigns **Question IDs** (`Q001` through `Q100`)
- Tags each question as `"Answerable"` or `"Unanswerable"`
- Combines them into a single structured DataFrame
- Prints the full set so you can verify formatting before saving

You can later export this to CSV for use in LLM evaluation pipelines, RAG experiments, or chatbot grounding tests.


In [ ]:
import pandas as pd

# 50 answerable questions
answerable_questions = [
    "Can you manage diabetes just with food and no meds?",
    "Is walking after meals actually helpful for blood sugar?",
    "Do probiotics help with blood sugar control?",
    "Should I avoid bananas or are they okay in moderation?",
    "Can I still eat carbs if I'm trying to lower my A1C?",
    "How many grams of sugar should I aim for in a day if I’m prediabetic?",
    "What’s the best breakfast for someone with type 2 diabetes?",
    "Are low-carb diets the only way to manage diabetes?",
    "What is the best way to manage cravings?",
    "Can I still eat dessert if I plan my meals right?",
    "Can drinking alcohol mess with my blood sugar readings?",
    "Is oatmeal okay or does it spike blood sugar too much?",
    "Are cheat days okay if I eat healthy most of the week?",
    "Should I eat the same number of carbs at every meal?",
    "Can weight loss actually reverse prediabetes?",
    "What should I do if I feel dizzy or shaky between meals?",
    "How can I prevent diabetes complications like nerve damage?",
    "Do I need to count net carbs or total carbs?",
    "Is eating out possible, or should I always cook at home?",
    "How long does it take to see A1C improvement after changing your diet?",
    "Can I eat pasta if I pair it with vegetables and protein?",
    "Is coffee bad for blood sugar control?",
    "Do protein shakes spike blood sugar?",
    "How does fiber help with blood sugar?",
    "How can I talk to my family about changing our eating habits?",
    "Do I need to take supplements like magnesium or cinnamon?",
    "What’s a safe blood sugar range after eating?",
    "Are sugar substitutes safe to use daily?",
    "Are CGMs worth it for someone who’s not on insulin?",
    "Do I have to give up rice completely?",
    "How much water should I drink each day to help manage blood sugar?",
    "Can skipping meals make my blood sugar worse?",
    "Can high cholesterol make diabetes worse?",
    "Are there smartphone apps that actually help with managing diabetes?",
    "Do I need to see a nutritionist or can I figure this out on my own?",
    "If my fasting glucose is okay but my A1C is high, what does that mean?",
    "What’s the difference between type 1 and type 2 diabetes?",
    "Are there cultural foods I can still eat while managing diabetes?",
    "What is the role of weight loss in achieving diabetes remission?",
    "What’s the best type of exercise for people with diabetes?",
    "Can I eat fruit if I watch my portions?",
    "Is brown rice better than white rice for blood sugar?",
    "How do I handle cravings for sweets in the evening?",
    "Can I eat bread if it's whole grain?",
    "How does stress affect my blood sugar levels?",
    "What kind of doctor should I see for prediabetes?",
    "Are there any easy meal prep ideas for managing diabetes?",
    "How much exercise per week helps with insulin resistance?",
    "Is it okay to drink diet soda if I'm diabetic?",
    "What is insulin resistance and how do I know if I have it?"
]

# 50 unanswerable questions
unanswerable_questions = [
    "Can I use apple cider vinegar to never get diabetes again?",
    "If I manifest better health, will my A1C improve?",
    "Can grounding (walking barefoot) balance my insulin levels?",
    "Will prayer alone replace insulin?",
    "Can eating only raw foods reverse diabetes permanently?",
    "If I cut out all sugar forever, will I become immune to diabetes?",
    "Can dreams predict blood sugar spikes?",
    "If I move to a tropical island, will my diabetes disappear?",
    "Is there one superfood that prevents diabetes?",
    "Can positive affirmations change my glucose curve?",
    "Will talking to plants reduce my cravings?",
    "Is diabetes punishment for past bad eating?",
    "Can I detox my pancreas by drinking lemon water for a week?",
    "If I fast for 3 days straight, will my diabetes reset?",
    "Will being vegan make me invincible to blood sugar issues?",
    "Is diabetes a government conspiracy?",
    "If I lose enough weight, can I delete my diabetes genes?",
    "Will getting rid of my microwave lower my A1C?",
    "Can I pass prediabetes to my kids through my bloodline?",
    "Is eating at the exact same time every day a diabetes cure?",
    "Can astrology tell me if I’ll get diabetes?",
    "Can sound therapy lower my blood sugar?",
    "Is wearing blue clothes good for my pancreas?",
    "If I walk backwards, will that improve insulin function?",
    "Can I train my body to not respond to carbs?",
    "Is prediabetes just made up by pharma to sell drugs?",
    "Will the new diabetes vaccine cure me instantly?",
    "Can aliens help us fix diabetes with their tech?",
    "Will a cold shower fix insulin resistance?",
    "Can I heal my beta cells by thinking positively?",
    "Is sugar evil and should I fear it like poison?",
    "Can I cure diabetes with essential oils?",
    "If I sleep 10 hours a night, will I reverse diabetes?",
    "Will breathing exercises change my glucose levels permanently?",
    "Is there a tea that makes insulin unnecessary?",
    "Will moving to the mountains make me diabetes-free?",
    "If my mom had gestational diabetes, does that mean I’m doomed?",
    "Can eating only purple food reverse prediabetes?",
    "Can I vibe my way to healthy blood sugar?",
    "Can dancing daily replace metformin?",
    "Can sunlight cure diabetes?",
    "Does singing before meals reduce insulin spikes?",
    "Can I permanently fix my pancreas with smoothies?",
    "Is there a moon phase that lowers A1C?",
    "Will avoiding shoes improve my glucose levels?",
    "Can standing on one leg daily reverse insulin resistance?",
    "If I only eat cold foods, will I avoid diabetes?",
    "Is diabetes caused by bad karma?",
    "Can my pet's diet affect my diabetes?",
    "Will thinking about sugar cause a spike in my blood glucose?"
]

# Combine questions and types
all_questions = answerable_questions + unanswerable_questions
question_types = ["Answerable"] * len(answerable_questions) + ["Unanswerable"] * len(unanswerable_questions)

# Generate question IDs: Q001 to Q100
question_ids = [f"Q{str(i+1).zfill(3)}" for i in range(len(all_questions))]

# Build DataFrame with Question ID
df = pd.DataFrame({
    "Question ID": question_ids,
    "Question Type": question_types,
    "Question": all_questions
})

# Display first 100 rows
print(df.head(100))

   Question ID Question Type  \
0         Q001    Answerable   
1         Q002    Answerable   
2         Q003    Answerable   
3         Q004    Answerable   
4         Q005    Answerable   
..         ...           ...   
95        Q096  Unanswerable   
96        Q097  Unanswerable   
97        Q098  Unanswerable   
98        Q099  Unanswerable   
99        Q100  Unanswerable   

                                             Question  
0   Can you manage diabetes just with food and no ...  
1   Is walking after meals actually helpful for bl...  
2        Do probiotics help with blood sugar control?  
3   Should I avoid bananas or are they okay in mod...  
4   Can I still eat carbs if I'm trying to lower m...  
..                                                ...  
95  Can standing on one leg daily reverse insulin ...  
96   If I only eat cold foods, will I avoid diabetes?  
97                   Is diabetes caused by bad karma?  
98              Can my pet's diet affect my diabetes?  

In [10]:
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatAnthropic, ChatGooglePalm
from langchain_community.chat_models import ChatOllama
from langchain.schema import HumanMessage, SystemMessage

def get_answer(question, model_name="gpt-4"):
    system_prompt = (
        "You are a friendly, empathetic assistant helping people with diabetes "
        "or prediabetes. Provide clear, supportive, evidence-based answers in plain language."
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Q: {question}\nA:")
    ]

    if model_name == "gpt-4":
        llm = ChatOpenAI(model="gpt-4", temperature=0)
    elif model_name == "gpt-3.5":
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    elif model_name == "claude-3":
        llm = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)
    elif model_name == "gemini-pro":
        llm = ChatGooglePalm(model="models/chat-bison-001", temperature=0)
    elif model_name == "gemma3":
        llm = ChatOllama(model="gemma:3b", temperature=0)  # adjust to your pulled model name
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return llm(messages).content.strip()

In [11]:
import requests
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatAnthropic, ChatGooglePalm
from langchain.schema import HumanMessage, SystemMessage

OLLAMA_API_URL = "http://zitro-box-1:11434/api/generate"

def get_answer(question, model_name="gpt-4"):
    system_prompt = (
        "You are a friendly, empathetic assistant helping people with diabetes "
        "or prediabetes. Provide clear, supportive, evidence-based answers in plain language."
    )

    if model_name == "gemma3":
        # Use direct Ollama HTTP request
        prompt = f"{system_prompt}\n\nQ: {question}\nA:"
        payload = {
            "model": "gemma3:4b-it-qat",  # adjust if using a different model name
            "prompt": prompt,
            "stream": False
        }

        response = requests.post(OLLAMA_API_URL, json=payload)
        if response.ok:
            return response.json()["response"].strip()
        else:
            raise RuntimeError(f"Ollama error: {response.status_code} - {response.text}")

    else:
        # Use LangChain LLMs
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Q: {question}\nA:")
        ]

        if model_name == "gpt-4":
            llm = ChatOpenAI(model="gpt-4", temperature=0)
        elif model_name == "gpt-3.5":
            llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        elif model_name == "claude-3":
            llm = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)
        elif model_name == "gemini-pro":
            llm = ChatGooglePalm(model="models/chat-bison-001", temperature=0)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

        return llm(messages).content.strip()


In [12]:
print(get_answer("Can I still eat dessert if I plan my meals right?", model_name="gemma3"))


ConnectTimeout: HTTPConnectionPool(host='zitro-box-1', port=11434): Max retries exceeded with url: /api/generate (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x10ce0d400>, 'Connection to zitro-box-1 timed out. (connect timeout=None)'))

In [ ]:
import pandas as pd
import time
from datetime import timedelta

# Track results and timing
results = []
total = len(df)

start_time = time.time()

print(f"🚀 Starting processing of {total} questions...\n")

for i, row in df.iterrows():
    question = row["Question"]
    qtype = row["Question Type"]
    qid = row["Question ID"]

    current_index = i + 1
    print(f"[{current_index}/{total}] ⏳ Processing {qid} ({qtype}): \"{question[:60]}...\"")

    try:
        answer = get_answer(question, model_name="gemma3")
    except Exception as e:
        answer = f"Error: {e}"

    results.append({
        "Question ID": qid,
        "Question Type": qtype,
        "Question": question,
        "Answer": answer
    })

    # Timing calculations
    elapsed = time.time() - start_time
    avg_time = elapsed / current_index
    remaining = total - current_index
    eta = remaining * avg_time
    total_est = avg_time * total

    print(
        f"✅ {qid} done.\n"
        f"   Elapsed: {timedelta(seconds=int(elapsed))} | "
        f"Remaining: {timedelta(seconds=int(eta))} | "
        f"Est. Total: {timedelta(seconds=int(total_est))}\n"
    )

# Save results to CSV
output_df = pd.DataFrame(results)
output_df.to_csv("diabetes_qna_gemma3.csv", index=False)

total_time = time.time() - start_time
print(f"🎉 All done in {timedelta(seconds=int(total_time))}! Results saved to diabetes_qna_gemma3.csv.")


🚀 Starting processing of 100 questions...

[1/100] ⏳ Processing Q001 (Answerable): "Can you manage diabetes just with food and no meds?..."
✅ Q001 done.
   Elapsed: 0:01:07 | Remaining: 1:51:41 | Est. Total: 1:52:49

[2/100] ⏳ Processing Q002 (Answerable): "Is walking after meals actually helpful for blood sugar?..."
✅ Q002 done.
   Elapsed: 0:02:11 | Remaining: 1:47:35 | Est. Total: 1:49:47

[3/100] ⏳ Processing Q003 (Answerable): "Do probiotics help with blood sugar control?..."
✅ Q003 done.
   Elapsed: 0:03:28 | Remaining: 1:52:22 | Est. Total: 1:55:50

[4/100] ⏳ Processing Q004 (Answerable): "Should I avoid bananas or are they okay in moderation?..."
✅ Q004 done.
   Elapsed: 0:04:30 | Remaining: 1:48:08 | Est. Total: 1:52:39

[5/100] ⏳ Processing Q005 (Answerable): "Can I still eat carbs if I'm trying to lower my A1C?..."
✅ Q005 done.
   Elapsed: 0:05:36 | Remaining: 1:46:40 | Est. Total: 1:52:17

[6/100] ⏳ Processing Q006 (Answerable): "How many grams of sugar should I aim for in 

In [ ]:
import pandas as pd

# Run questions through Gemma3 and store in list
results = []

for i, row in df.iterrows():
    question = row["Question"]
    qtype = row["Question Type"]
    qid = row["Question ID"]
    
    try:
        answer = get_answer(question, model_name="gemma3")
    except Exception as e:
        answer = f"Error: {e}"
    
    results.append({"Question ID": qid, "Question Type": qtype, "Question": question, "Answer": answer})

# Convert to DataFrame and save as CSV
output_df = pd.DataFrame(results)
output_df.to_csv("diabetes_qna_gemma3.csv", index=False)
